# MyFirstNEURON — Colab Edition (Prototype 1)

This notebook is a Google-Colab-friendly, Python/Jupyter port of **MyFirstNEURON**, a NEURON demo
by Arthur Houweling and Terry Sejnowski (Salk Institute), based on experiments from
*Electrophysiology of the Neuron* by Huguenard & McCormick. Original files:
https://modeldb.science/3808

**Status:** first working increment, covering Experiment 5/6 ("impulse generation" — a
Hodgkin-Huxley action potential in a single-compartment cell). Once this is confirmed working,
the remaining experiments will be added the same way.

No local installation is needed — just run the cells top to bottom in Colab.

## 1. Setup (run once per Colab session)

Installs the `neuron` Python package, clones the original `.mod` mechanism files from the
[ModelDB GitHub mirror](https://github.com/ModelDBRepository/3808), and compiles them with
`nrnivmodl`.

In [ ]:
%%capture
!pip install neuron

In [ ]:
import os

MOD_SRC_DIR = "mfn_src"

if not os.path.isdir(MOD_SRC_DIR):
    !git clone --depth 1 https://github.com/ModelDBRepository/3808.git {MOD_SRC_DIR}

!cd {MOD_SRC_DIR} && nrnivmodl

In [ ]:
from neuron import h
from neuron import load_mechanisms
import matplotlib.pyplot as plt

load_mechanisms(MOD_SRC_DIR)
h.load_file("stdrun.hoc")

print("NEURON is ready, mechanisms loaded from:", MOD_SRC_DIR)

## 2. Build the cell

A single spherical compartment (same geometry as the original demo: total membrane area of
29000 &mu;m&sup2;), with passive leak channels (Na/K/Ca/Cl/Mg) and Hodgkin-Huxley
sodium/potassium channels.

In [ ]:
import math

soma = h.Section(name="soma")
soma.L = 290 / math.pi
soma.diam = 100
soma.nseg = 1

soma.insert("leak")
soma.insert("HH")

h.celsius = 35

# ionic concentrations (mM), as in the original e5.par
soma(0.5).nai = 31
soma(0.5).nao = 145
soma(0.5).ki = 135
soma(0.5).ko = 3.1

print("soma area (um^2):", h.area(0.5, sec=soma))

## 3. Stimulus and recording

In [ ]:
stim = h.IClamp(soma(0.5))

t_vec = h.Vector().record(h._ref_t)
v_vec = h.Vector().record(soma(0.5)._ref_v)

## 4. Run once (sanity check)

Parameters below match the original Experiment 5 (`e5.par`): a 50 ms, 2 nA current step
should evoke a train of action potentials.

In [ ]:
def run_experiment(gnabar_HH=0.069, gkbar_HH=0.0069, pna_leak=2.07e-7, pk_leak=3.45e-6,
                    stim_delay=10, stim_dur=50, stim_amp=2, tstop=80, v_init=-65, dt=0.05):
    soma(0.5).gnabar_HH = gnabar_HH
    soma(0.5).gkbar_HH = gkbar_HH
    soma(0.5).pna_leak = pna_leak
    soma(0.5).pk_leak = pk_leak

    stim.delay = stim_delay
    stim.dur = stim_dur
    stim.amp = stim_amp

    h.tstop = tstop
    h.v_init = v_init
    h.dt = dt
    h.run()

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(t_vec, v_vec)
    ax.set_xlabel("time (ms)")
    ax.set_ylabel("membrane potential (mV)")
    ax.set_title("Experiment 5/6: impulse generation")
    ax.set_ylim(-100, 50)
    plt.show()

run_experiment()

## 5. Interactive exploration

Adjust the sliders, then click **Run** to simulate. Each parameter
has a **reset** button to restore its default, and a **changed** checkbox that ticks itself
whenever the value differs from default.

In [ ]:
from ipywidgets import FloatSlider, Checkbox, Button, HBox, VBox, Output, Layout
from IPython.display import display

# single source of truth for defaults, ranges and slider formatting
PARAM_SPECS = {
    "gnabar_HH": dict(default=0.069, min=0, max=0.2, step=0.001, readout_format=".3f", description="gNa (HH)"),
    "gkbar_HH": dict(default=0.0069, min=0, max=0.02, step=0.0005, readout_format=".4f", description="gK (HH)"),
    "pna_leak": dict(default=2.07e-7, min=0, max=1e-6, step=1e-8, readout_format=".2e", description="pNa (leak)"),
    "pk_leak": dict(default=3.45e-6, min=0, max=1e-5, step=1e-7, readout_format=".2e", description="pK (leak)"),
    "stim_delay": dict(default=10, min=0, max=50, step=1, description="stim delay (ms)"),
    "stim_dur": dict(default=50, min=1, max=100, step=1, description="stim dur (ms)"),
    "stim_amp": dict(default=2, min=0, max=5, step=0.1, description="stim amp (nA)"),
    "tstop": dict(default=80, min=20, max=200, step=5, description="tstop (ms)"),
    "v_init": dict(default=-65, min=-90, max=-40, step=1, description="v_init (mV)"),
    "dt": dict(default=0.05, min=0.01, max=0.25, step=0.01, description="dt (ms)"),
}

sliders = {}
changed_flags = {}
rows = []

def _make_change_handler(name):
    def _on_change(change):
        changed_flags[name].value = (change["new"] != PARAM_SPECS[name]["default"])
    return _on_change

def _make_reset_handler(name):
    def _on_click(_btn):
        sliders[name].value = PARAM_SPECS[name]["default"]
    return _on_click

for name, spec in PARAM_SPECS.items():
    slider_kwargs = {k: v for k, v in spec.items() if k != "default"}
    slider = FloatSlider(value=spec["default"], layout=Layout(width="350px"), **slider_kwargs)
    changed = Checkbox(value=False, description="changed", disabled=True, indent=False,
                        layout=Layout(width="90px"))
    reset_btn = Button(description="reset", layout=Layout(width="60px"))

    sliders[name] = slider
    changed_flags[name] = changed
    slider.observe(_make_change_handler(name), names="value")
    reset_btn.on_click(_make_reset_handler(name))

    rows.append(HBox([slider, changed, reset_btn]))

run_button = Button(description="Run", button_style="success", icon="play")
reset_all_button = Button(description="Reset all", icon="undo")
output = Output()

def _on_run(_btn):
    with output:
        output.clear_output(wait=True)
        values = {name: slider.value for name, slider in sliders.items()}
        run_experiment(**values)

def _on_reset_all(_btn):
    for name, slider in sliders.items():
        slider.value = PARAM_SPECS[name]["default"]

run_button.on_click(_on_run)
reset_all_button.on_click(_on_reset_all)

display(VBox(rows + [HBox([run_button, reset_all_button]), output]))